# 02 - Exploratory Data Analysis

**Prerequisites:** Run `01_data_discovery.ipynb` first and complete data download.

**Purpose:** Understand the downloaded dataset before feature engineering:
- Distribution of cases by year, court, case type
- Judge distribution and activity patterns
- Text length and quality distributions
- Missing data patterns
- Initial outcome signal exploration

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from configs.settings import settings
from src.data.validators import TausiDecisionRaw, DatasetStats

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Downloaded Data

In [ ]:
def load_all_cases(raw_dir: Path = settings.data.raw_dir):
    """Load all downloaded JSON case files."""
    cases = []
    for json_path in sorted(raw_dir.rglob('*.json')):
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        try:
            case = TausiDecisionRaw.model_validate(data)
            cases.append(case)
        except Exception as e:
            pass  # Skip invalid cases
    return cases

cases = load_all_cases()
print(f'Loaded {len(cases)} cases')

## 2. Year Distribution

In [ ]:
years = [c.filing_year for c in cases if c.filing_year]
fig, ax = plt.subplots(figsize=(10, 5))
pd.Series(years).value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Cases by Filing Year')
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Case Type Distribution

In [ ]:
case_types = [c.casetype.get('name', 'Unknown') if c.casetype else 'Unknown' for c in cases]
ct_series = pd.Series(case_types).value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
ct_series.plot(kind='barh', ax=ax)
ax.set_title('Top 15 Case Types')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 4. Judge Activity

In [ ]:
all_judges = []
for c in cases:
    if c.judges:
        primary = c.judge_names[0] if c.judge_names else 'Unknown'
        all_judges.append(primary)

judge_counts = pd.Series(all_judges).value_counts()
print(f'Total unique judges: {len(judge_counts)}')
print(f'Judges with >= 15 cases: {(judge_counts >= 15).sum()}')
print(f'Judges with >= 30 cases: {(judge_counts >= 30).sum()}')
print(f'\nTop 20 most active judges:')
print(judge_counts.head(20))

## 5. Field Completeness Summary

In [ ]:
stats = DatasetStats(
    total_cases=len(cases),
    cases_with_judges=sum(1 for c in cases if c.judges),
    cases_with_advocates=sum(1 for c in cases if c.advocates),
    cases_with_citations=sum(1 for c in cases if c.cited_documents),
    year_distribution=dict(pd.Series(years).value_counts()),
)
print(stats.completeness_report)

## 6. Text Length Distribution

Check extracted text quality (run `python -m src.data.pdf_extractor` first).

In [ ]:
text_lengths = []
for txt_path in settings.data.interim_dir.rglob('*.txt'):
    text_lengths.append(txt_path.stat().st_size)

if text_lengths:
    lengths_series = pd.Series(text_lengths)
    print(f'Extracted texts: {len(text_lengths)}')
    print(lengths_series.describe())
    
    fig, ax = plt.subplots(figsize=(10, 5))
    lengths_series.clip(upper=100000).hist(bins=50, ax=ax)
    ax.set_title('Judgment Text Length Distribution')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No extracted texts found. Run pdf_extractor first.')